# Priority — full bake-off

Benchmarks every **regime x language x model x arm** combination on **dev**, then
picks a winner and saves its predictions.

Headline metric: **`macro_f1`** (macro-F1 over Low/Medium/High).

**Why not accuracy.** `Low` is 53% of tickets and `High` is 10%. Accuracy hides a collapse on `High`, which is the class with the operational cost.

Test stays sealed. Nothing in this notebook touches it.

In [ ]:
import sys, time
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import pandas as pd
import matplotlib.pyplot as plt

import swiftbench as sb

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

AUTHOR = ""

manifest = sb.splits.ensure()
print("split sha:", manifest["sha"], manifest["counts"])
print("models:", sb.models.NAMES)
print("arms:  ", sb.imbalance.ARMS)
print("regimes:", list(sb.train_classical.REGIMES))

## The three training regimes

This is the deployment question, not just a hyperparameter. Do we ship one
multilingual model or five monolingual ones?

| regime | trains on | answers |
|---|---|---|
| `mono` | the evaluation language only | how good can a dedicated per-language model get? |
| `multi` | all five languages at once | does one model serve everything without losing accuracy? |
| `zeroshot-en` | English only, never the eval language | how much transfers across scripts for free? |

`zeroshot-en` is the one that says anything about a language we have not built a
dataset for. Its English row is skipped — training on English and evaluating on
English is not zero-shot.

## 1. Model-free floors

Every trained number below has to be read against these.

In [ ]:
TASK = "priority"

floor_rows = []
for lang in sb.config.LANGUAGES:
    tr, dv = sb.splits.get([lang], "train"), sb.splits.get([lang], "dev")
    for name, s in {
        "majority": sb.baselines.majority(tr, dv, TASK),
        "intent-lookup-oracle": sb.baselines.intent_lookup(tr, dv, TASK),
        "intent-chained": sb.baselines.intent_chained(tr, dv, TASK),
    }.items():
        floor_rows.append({"language": lang, "model": name, "arm": "none",
                           "regime": "mono", **{k: v for k, v in s.items() if not k.startswith("_")}})
floor_df = pd.DataFrame(floor_rows)
display(floor_df.pivot_table(index="language", columns="model", values="macro_f1").round(4))
print("intent-chained is THE BAR (predicted intent -> lookup).")
print("intent-lookup-oracle uses GOLD intent — an upper bound, never a target.")
CHAINED = floor_df[floor_df.model == "intent-chained"].set_index("language")["macro_f1"]
ORACLE = floor_df[floor_df.model == "intent-lookup-oracle"].set_index("language")["macro_f1"]

## 2. The sweep

4 models x 3 arms x 3 regimes across 5 languages. `multi` and `zeroshot-en`
fit once per (training set, model, arm) and are then evaluated against each
language, so this is far fewer fits than the grid size suggests.

Progress prints per regime. Expect several minutes — the `multi` fits train on
42,500 rows.

In [ ]:
started = time.time()
sweep = sb.train_classical.sweep_regimes(task=TASK, author=AUTHOR)
print(f"\n{len(sweep)} evaluations in {time.time() - started:.0f}s")

sweep.to_csv(REPO / "ml" / "reports" / f"{TASK}_bakeoff_dev.csv", index=False)
sweep.head()

## 3. Which regime wins?

Averaged over languages, models and arms — the top-level deployment answer.

In [ ]:
by_regime = (sweep.groupby("regime")[["macro_f1", "accuracy", "macro_f1"]]
             .agg(["mean", "max"]).round(4))
display(by_regime)

print("\nBest single configuration per regime:")
best_per_regime = sweep.loc[sweep.groupby("regime")["macro_f1"].idxmax()]
display(best_per_regime[["regime", "language", "model", "arm", "macro_f1",
                         "accuracy", "n_train"]].round(4))

## 4. Which model and which arm?

The arm usually matters more than the estimator. Check that before spending
effort on model choice.

In [ ]:
print("by model (mean macro_f1 across everything):")
display(sweep.groupby("model")["macro_f1"].agg(["mean", "max"]).round(4)
        .sort_values("mean", ascending=False))

print("\nby arm:")
display(sweep.groupby("arm")[["macro_f1", "accuracy"]].mean().round(4))

print("\nmodel x arm, mono regime only:")
mono = sweep[sweep.regime == "mono"]
display(mono.pivot_table(index="model", columns="arm", values="macro_f1").round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

pv = sweep.pivot_table(index="model", columns="regime", values="macro_f1")
pv.plot(kind="bar", ax=axes[0], rot=0, colormap="viridis")
axes[0].axhline(CHAINED.mean(), color="#c44", ls="--", lw=1.4,
                label=f"intent-chained bar ({CHAINED.mean():.3f})")
axes[0].axhline(ORACLE.mean(), color="#999", ls=":", lw=1.4,
                label=f"oracle ({ORACLE.mean():.3f})")
axes[0].set_title("Priority macro-F1 by model and regime")
axes[0].set_ylabel("macro_f1 (dev)")
axes[0].set_ylim(0.7, 1.0)
axes[0].legend(fontsize=8)

# does High survive?
per_class = sweep.groupby("arm")[["f1_low", "f1_medium", "f1_high"]].mean()
per_class.plot(kind="bar", ax=axes[1], rot=0, color=["#8bc", "#59a", "#c44"])
axes[1].set_title("Per-class F1 by arm — does High collapse?")
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Per-language winner

What we would actually ship if we shipped per-language models.

In [ ]:
winners = sweep.loc[sweep.groupby("language")["macro_f1"].idxmax()]
display(winners[["language", "regime", "model", "arm", "macro_f1",
                 "accuracy", "macro_f1"]].round(4).sort_values("macro_f1", ascending=False))

# how much does forcing a single multilingual model cost us, per language?
best_mono = sweep[sweep.regime == "mono"].groupby("language")["macro_f1"].max()
best_multi = sweep[sweep.regime == "multi"].groupby("language")["macro_f1"].max()
best_zs = sweep[sweep.regime == "zeroshot-en"].groupby("language")["macro_f1"].max()
cmp = pd.DataFrame({"mono": best_mono, "multi": best_multi, "zeroshot_en": best_zs})
cmp["multi_minus_mono"] = (cmp["multi"] - cmp["mono"]).round(4)
display(cmp.round(4))

## 6. Champion — predictions and error analysis

Refit the single best configuration, save its dev predictions, and look at what
it still gets wrong.

In [ ]:
champ = sweep.loc[sweep["macro_f1"].idxmax()]
train_langs = sb.train_classical.REGIMES[champ.regime](champ.language)
print(f"champion: {champ.model} / arm={champ.arm} / regime={champ.regime} "
      f"/ eval={champ.language}  ->  macro_f1 {champ['macro_f1']:.4f}")

fit = sb.train_classical.run(TASK, champ.model, train_langs, champ.language,
                             arm=champ.arm, save=False, verbose=False)
dv = sb.splits.get([champ.language], "dev")

pred_dir = REPO / "ml" / "predictions"
pred_dir.mkdir(parents=True, exist_ok=True)
out = dv[["id", "language", "text_en", "text", "category", TASK]].copy()
out[f"predicted_{TASK}"] = fit["_predictions"]
out["correct"] = out[TASK] == out[f"predicted_{TASK}"]
out.to_csv(pred_dir / f"{TASK}_champion_dev_predictions.csv", index=False)
print(f"saved {len(out)} predictions -> ml/predictions/{TASK}_champion_dev_predictions.csv")

labels, cm = sb.metrics.confusion(dv[TASK], fit["_predictions"], TASK)
display(pd.DataFrame(cm, index=[f"true {l}" for l in labels],
                     columns=[f"pred {l}" for l in labels]))

In [ ]:
errors = out[~out.correct]
print(f"{len(errors)} errors out of {len(out)} ({len(errors)/len(out):.1%})\n")
print("most confused pairs:")
display(errors.groupby([TASK, f"predicted_{TASK}"]).size()
        .sort_values(ascending=False).head(8).rename("count").reset_index())
print("\nsample errors:")
for _, r in errors.head(8).iterrows():
    print(f"  true={r[TASK]:8s} pred={r[f'predicted_{TASK}']:8s} [{r.category}] {str(r.text_en)[:80]}")

## 7. Read-out

- Majority (`Low` everywhere): macro-F1 0.230
- `intent-chained` — **the bar a direct model must beat to justify existing**
- `intent-lookup-oracle` — upper bound only
- Champion: see above

If the champion clears `intent-chained`, priority gets its own head and stops
inheriting the intent classifier's errors. If it does not, priority is a lookup
hung off the intent model and there is no second model to train.

Reminder: this is `priority_base`, derived from ticket content alone. Serving
priority also needs wait time and queue state, which live in the serving
pipeline and are deliberately not in this dataset.